In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [22]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [23]:
len(docs)

9

In [24]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)

In [25]:
len(splitted_docs)

26

In [26]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")
vector_store = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

### agent = tools, llm, prompt

In [27]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the relevant data of the PDF Documents, and these pdf
        documents have details about medical reports.
    """
    print("tool called:", query)
    docs = vector_store.similarity_search(query=query, k=4)
    context =""

    for doc in docs:
        context = doc.page_content + "\n\n"
    return context
    

In [28]:
from langchain_groq import ChatGroq
import os
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.environ.get("GROQ_API_KEY"))

In [29]:
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever_tool` tool for questions requiring external knowledge.
"""

In [30]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_Prompt
)

In [31]:
query = "what is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})

tool called: patient name doctor name


In [32]:
result = response["messages"][-1].content

In [33]:
print(result)

**Patient Name:** Ms. Nikita Chudhary  
**Doctor’s Name:** Dr. Nitin Nahar
